In [ ]:
import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

def iniciar_navegador():
    options = Options()
    options.add_argument("--headless")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("window-size=1920,1080")
    options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36")
    return webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

# URL DIRECTA AL DASHBOARD (Para saltar la página de bienvenida)
URL_DASHBOARD = "https://reportes.odepa.gob.cl/#/precio-volumen-diario-fruta-hortaliza"

driver = iniciar_navegador()
dataset_final = []

try:
    print("🚀 Saltando portada y entrando al Dashboard...")
    driver.get(URL_DASHBOARD)
    
    # Espera larga inicial para que el dashboard de Power BI / JS cargue
    wait = WebDriverWait(driver, 30)
    
    # --- PASO 1: LOCALIZAR FILTROS ---
    print("📍 Buscando filtros de Región...")
    # Buscamos el div que contiene el texto de selección
    filtro_region = wait.until(EC.element_to_be_clickable((By.XPATH, "//div[contains(@class, 'form-group')]//div[contains(text(), 'Región')]")))
    driver.execute_script("arguments[0].click();", filtro_region) # Clic via JS es más fiable aquí
    
    print("✅ Seleccionando Coquimbo...")
    opcion_coquimbo = wait.until(EC.element_to_be_clickable((By.XPATH, "//span[contains(text(), 'Coquimbo')]")))
    opcion_coquimbo.click()

    # --- PASO 2: MERCADO ---
    print("🏪 Seleccionando Terminal La Palmera...")
    filtro_mercado = wait.until(EC.element_to_be_clickable((By.XPATH, "//*[contains(text(), 'mercado')]")))
    driver.execute_script("arguments[0].click();", filtro_mercado)
    
    opcion_palmera = wait.until(EC.element_to_be_clickable((By.XPATH, "//*[contains(text(), 'Terminal La Palmera')]")))
    opcion_palmera.click()

    # --- PASO 3: EXTRACCIÓN ---
    print("📊 Esperando datos de la tabla...")
    time.sleep(5) # Pausa técnica para renderizado
    
    # Buscamos las filas de la tabla
    wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "table tbody tr")))
    filas = driver.find_elements(By.CSS_SELECTOR, "table tbody tr")

    for fila in filas:
        try:
            cols = fila.find_elements(By.TAG_NAME, "td")
            if len(cols) >= 6:
                dataset_final.append({
                    "Variedad": cols[0].text,
                    "Calidad": cols[1].text,
                    "Volumen": cols[2].text.replace(".", ""),
                    "Precio_Pond": cols[5].text.replace(".", ""),
                    "Unidad": cols[6].text
                })
        except:
            continue

    if dataset_final:
        df = pd.DataFrame(dataset_final)
        df.to_csv("datos_coquimbo_exito.csv", index=False, encoding='utf-8-sig')
        print(f"✨ ¡Conseguido! {len(df)} filas extraídas.")
        print(df.head())
    else:
        print("⚠ Tabla encontrada pero vacía. Revisa si hay datos para hoy.")

except Exception as e:
    print(f"❌ Error: {e}")
    driver.save_screenshot("error_final_debug.png")
    # Capturamos el código fuente para analizar qué etiquetas hay realmente
    with open("source.html", "w") as f:
        f.write(driver.page_source)
    print("📸 Revisa 'error_final_debug.png' y 'source.html'.")

finally:
    driver.quit()

🚀 Saltando portada y entrando al Dashboard...
📍 Buscando filtros de Región...
